# Masked Pokemon Team Transformer

Predicts a masked team member (species, ability, item, moveset) given the other 5 Pokemon on a team.

Each Pokemon is encoded as:

`Species Learned Emb | Species RDV | Ability Learned Emb | Item Learned Emb | Item RDV | Moveset Learned Emb | Moveset RDV`

(Ability has only a learned embedding; no ability RDV was prepared.) The moveset embedding/RDV are the
mean across the Pokemon's 4 moves, with an optional multi-head attention layer applied over the 4 moves
before averaging.

In [1]:
import pickle
import random
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Configuration

Edit this cell to change embedding sizes, transformer size, masking, and training settings.

In [2]:
# ---- Learned-embedding size presets (species / ability / item / move) ----
EMBED_CONFIGS = {
    "small":  {"species": 16, "ability": 8,  "item": 8,  "move": 16},
    "medium": {"species": 32, "ability": 16, "item": 16, "move": 32},
    "large":  {"species": 64, "ability": 32, "item": 32, "move": 64},
}
EMBED_CONFIG_NAME = "small"          # one of: small | medium | large
EMB = EMBED_CONFIGS[EMBED_CONFIG_NAME]

# ---- Raw-data-vector dimensionalities (fixed by the prepared .pkl files) ----
SPECIES_RDV_DIM = 42
ITEM_RDV_DIM    = 6
MOVE_RDV_DIM    = 44

# ---- Moveset aggregation ----
USE_MOVE_ATTENTION = False           # if True, run MHA over the 4 moves before averaging
MOVE_ATTENTION_HEADS = 2

# ---- Transformer ----
D_MODEL          = 128
N_HEADS          = 4
N_LAYERS         = 2
DIM_FEEDFORWARD  = 256
DROPOUT          = 0.1

# ---- Masking ----
TRAIN_MASK_STRATEGY = "random"       # "random" = mask any of the 6 (augmentation); "last" = always 6th
EVAL_MASK_STRATEGY  = "last"         # held-out evaluation always predicts the 6th slot

# ---- Training ----
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 1e-3
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Debug-run override (Section 8) ----
DEBUG_RUN          = True            # if True, use a tiny 50-team sample to smoke-test the pipeline
DEBUG_TOTAL_TEAMS  = 50
DEBUG_TEST_TEAMS   = 10

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)
print("Config:", EMBED_CONFIG_NAME, EMB, "| device:", DEVICE)

Config: small {'species': 16, 'ability': 8, 'item': 8, 'move': 16} | device: cpu


## 1. Data Loading

Raw-data-vector dictionaries (`pokemon_vectors.pkl`, `item_vectors.pkl`, `move_vectors.pkl`) come from
`prepare_raw_vectors.ipynb`. Team pools are the scraped `*_team_vectors*.pkl` files. Change `POOL_FILES`
to alter which sources feed the model.

In [3]:
with open("pokemon_vectors.pkl", "rb") as f: SPECIES_RDV = pickle.load(f)
with open("item_vectors.pkl",    "rb") as f: ITEM_RDV    = pickle.load(f)
with open("move_vectors.pkl",    "rb") as f: MOVE_RDV    = pickle.load(f)
with open("ability_dict.pkl",    "rb") as f: ABILITY_DICT = pickle.load(f)

# Each pool file -> list of teams; each team -> 6 Pokemon;
# each Pokemon -> [species, ability, item, move1, move2, move3, move4]
POOL_FILES = {
    "vgcpastes":  "team_vectors.pkl",
    "vgenc":      "vgenc_team_vectors.pkl",
    "limitless":  "limitless_team_vectors.pkl",
}
TEAM_POOLS = {}
for name, path in POOL_FILES.items():
    with open(path, "rb") as f:
        TEAM_POOLS[name] = pickle.load(f)
    print(f"{name:11s}: {len(TEAM_POOLS[name])} teams")

vgcpastes  : 769 teams
vgenc      : 2529 teams
limitless  : 6109 teams


## 2. Mega Stone Handling

If a Pokemon holds a Mega Stone, its species is renamed to the corresponding Mega form so it receives
the Mega's base stats / typing RDV. The held item is left unchanged. Charizard (X/Y) is handled
explicitly.

In [4]:
MEGA_STONE_TO_FORM = {
    "Abomasite": "Abomasnow-Mega", "Absolite": "Absol-Mega",
    "Aerodactylite": "Aerodactyl-Mega", "Aggronite": "Aggron-Mega",
    "Alakazite": "Alakazam-Mega", "Altarianite": "Altaria-Mega",
    "Ampharosite": "Ampharos-Mega", "Audinite": "Audino-Mega",
    "Banettite": "Banette-Mega", "Beedrillite": "Beedrill-Mega",
    "Blastoisinite": "Blastoise-Mega", "Cameruptite": "Camerupt-Mega",
    "Chandelurite": "Chandelure-Mega",
    "Charizardite X": "Charizard-Mega-X", "Charizardite Y": "Charizard-Mega-Y",
    "Chesnaughtite": "Chesnaught-Mega", "Chimechite": "Chimecho-Mega",
    "Clefablite": "Clefable-Mega", "Crabominite": "Crabominable-Mega",
    "Delphoxite": "Delphox-Mega", "Dragoninite": "Dragonite-Mega",
    "Drampanite": "Drampa-Mega", "Emboarite": "Emboar-Mega",
    "Excadrite": "Excadrill-Mega", "Feraligite": "Feraligatr-Mega",
    "Floettite": "Floette-Mega", "Froslassite": "Froslass-Mega",
    "Galladite": "Gallade-Mega", "Garchompite": "Garchomp-Mega",
    "Gardevoirite": "Gardevoir-Mega", "Gengarite": "Gengar-Mega",
    "Glalitite": "Glalie-Mega", "Glimmoranite": "Glimmora-Mega",
    "Golurkite": "Golurk-Mega", "Greninjite": "Greninja-Mega",
    "Gyaradosite": "Gyarados-Mega", "Hawluchanite": "Hawlucha-Mega",
    "Heracronite": "Heracross-Mega", "Houndoominite": "Houndoom-Mega",
    "Kangaskhanite": "Kangaskhan-Mega", "Lopunnite": "Lopunny-Mega",
    "Lucarionite": "Lucario-Mega", "Manectite": "Manectric-Mega",
    "Medichamite": "Medicham-Mega", "Meganiumite": "Meganium-Mega",
    "Meowsticite": "Meowstic-Mega", "Pidgeotite": "Pidgeot-Mega",
    "Pinsirite": "Pinsir-Mega", "Sablenite": "Sableye-Mega",
    "Scizorite": "Scizor-Mega", "Scovillainite": "Scovillain-Mega",
    "Sharpedonite": "Sharpedo-Mega", "Skarmorite": "Skarmory-Mega",
    "Slowbronite": "Slowbro-Mega", "Starminite": "Starmie-Mega",
    "Steelixite": "Steelix-Mega", "Tyranitarite": "Tyranitar-Mega",
    "Venusaurite": "Venusaur-Mega", "Victreebelite": "Victreebel-Mega",
}

def apply_mega(pokemon):
    """Return a copy of one Pokemon list with species renamed to its Mega form
    when it holds a Mega Stone (and that Mega form exists in the species RDVs)."""
    species, ability, item = pokemon[0], pokemon[1], pokemon[2]
    mega = MEGA_STONE_TO_FORM.get(item)
    if mega is not None and mega in SPECIES_RDV:
        p = list(pokemon)
        p[0] = mega
        return p
    return list(pokemon)

def normalize_team(team):
    return [apply_mega(p) for p in team]

## 3. Vocabularies

Learned-embedding vocabularies are built from every name observed across **all** pools (so train and
held-out teams share the index space) unioned with the RDV dictionary keys. Index 0 is reserved for
`<UNK>` (covers names with no RDV / unseen entries). RDV lookups fall back to a zero vector when a
name is absent, and missing fractions are reported.

In [5]:
UNK = "<UNK>"

def build_vocab(values):
    vocab = {UNK: 0}
    for v in values:
        if v not in vocab:
            vocab[v] = len(vocab)
    return vocab

_sp, _ab, _it, _mv = set(SPECIES_RDV), set(ABILITY_DICT), set(ITEM_RDV), set(MOVE_RDV)
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            _sp.add(p[0]); _ab.add(p[1]); _it.add(p[2])
            for m in p[3:7]: _mv.add(m)

SPECIES_VOCAB = build_vocab(sorted(_sp))
ABILITY_VOCAB = build_vocab(sorted(_ab))
ITEM_VOCAB    = build_vocab(sorted(_it))
MOVE_VOCAB    = build_vocab(sorted(_mv))
N_SPECIES, N_ABILITY = len(SPECIES_VOCAB), len(ABILITY_VOCAB)
N_ITEM, N_MOVE       = len(ITEM_VOCAB), len(MOVE_VOCAB)
print(f"vocab sizes -> species {N_SPECIES}, ability {N_ABILITY}, item {N_ITEM}, move {N_MOVE}")

ZERO_SPECIES_RDV = [0.0] * SPECIES_RDV_DIM
ZERO_ITEM_RDV    = [0.0] * ITEM_RDV_DIM
ZERO_MOVE_RDV    = [0.0] * MOVE_RDV_DIM

# RDV coverage diagnostic over normalized teams
tot = miss_s = miss_i = miss_m = 0
for pool in TEAM_POOLS.values():
    for team in pool:
        for p in normalize_team(team):
            tot += 1
            if p[0] not in SPECIES_RDV: miss_s += 1
            if p[2] not in ITEM_RDV:    miss_i += 1
            for m in p[3:7]:
                if m not in MOVE_RDV:   miss_m += 1
print(f"RDV miss rate -> species {miss_s/tot:.1%}, item {miss_i/tot:.1%}, move {miss_m/(tot*4):.1%}")

vocab sizes -> species 359, ability 269, item 209, move 682
RDV miss rate -> species 7.5%, item 0.9%, move 1.2%


## 4. Team -> Tensor Conversion

`team_to_tensors` maps a raw nested-list team (post Mega-rename) to index tensors and RDV tensors by
looking each name up in its vocabulary / RDV dictionary. `MaskedTeamDataset` then picks the masked
slot per sample and exposes the prediction targets.

In [6]:
def team_to_tensors(team):
    """team: list of 6 Pokemon lists (already Mega-normalized).
    Returns a dict of tensors describing all 6 Pokemon."""
    sp_idx, ab_idx, it_idx = [], [], []
    mv_idx, sp_rdv, it_rdv, mv_rdv = [], [], [], []
    for p in team:
        species, ability, item = p[0], p[1], p[2]
        moves = list(p[3:7]) + [""] * (4 - len(p[3:7]))
        sp_idx.append(SPECIES_VOCAB.get(species, 0))
        ab_idx.append(ABILITY_VOCAB.get(ability, 0))
        it_idx.append(ITEM_VOCAB.get(item, 0))
        mv_idx.append([MOVE_VOCAB.get(m, 0) for m in moves])
        sp_rdv.append(SPECIES_RDV.get(species, ZERO_SPECIES_RDV))
        it_rdv.append(ITEM_RDV.get(item, ZERO_ITEM_RDV))
        mv_rdv.append([MOVE_RDV.get(m, ZERO_MOVE_RDV) for m in moves])
    return {
        "species_idx": torch.tensor(sp_idx, dtype=torch.long),       # [6]
        "ability_idx": torch.tensor(ab_idx, dtype=torch.long),       # [6]
        "item_idx":    torch.tensor(it_idx, dtype=torch.long),       # [6]
        "move_idx":    torch.tensor(mv_idx, dtype=torch.long),       # [6,4]
        "species_rdv": torch.tensor(sp_rdv, dtype=torch.float),      # [6,42]
        "item_rdv":    torch.tensor(it_rdv, dtype=torch.float),      # [6,6]
        "move_rdv":    torch.tensor(mv_rdv, dtype=torch.float),      # [6,4,44]
    }

class MaskedTeamDataset(Dataset):
    def __init__(self, teams, mask_strategy):
        # store normalized teams
        self.teams = [normalize_team(t) for t in teams]
        self.mask_strategy = mask_strategy

    def __len__(self):
        return len(self.teams)

    def __getitem__(self, i):
        team = self.teams[i]
        t = team_to_tensors(team)
        if self.mask_strategy == "random":
            mpos = random.randrange(6)
        else:                       # "last"
            mpos = 5
        t["mask_pos"] = torch.tensor(mpos, dtype=torch.long)
        # targets = the masked Pokemon's attributes
        t["y_species"] = t["species_idx"][mpos].clone()
        t["y_ability"] = t["ability_idx"][mpos].clone()
        t["y_item"]    = t["item_idx"][mpos].clone()
        mv_multi = torch.zeros(N_MOVE)
        for mi in t["move_idx"][mpos].tolist():
            mv_multi[mi] = 1.0
        t["y_moves"] = mv_multi                                    # multi-hot moveset
        return t

## 5. Train / Test Split

Edit this section to change the pool or the split. By default the **test set is a random 20% of the
`limitless` teams**; everything else (`vgcpastes`, `vgenc`, and the remaining 80% of `limitless`)
is training. The debug override (Section 8) instead carves a 50-team sample.

In [7]:
TEST_SOURCE   = "limitless"
TEST_FRACTION = 0.20

def make_split(team_pools, test_source=TEST_SOURCE, test_fraction=TEST_FRACTION, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[test_source])
    idx = list(range(len(src)))
    rng.shuffle(idx)
    n_test = int(round(len(src) * test_fraction))
    test_ids = set(idx[:n_test])
    test_teams  = [src[i] for i in idx[:n_test]]
    train_teams = [src[i] for i in idx[n_test:]]
    for name, pool in team_pools.items():
        if name == test_source:
            continue
        train_teams.extend(pool)
    rng.shuffle(train_teams)
    return train_teams, test_teams

def make_debug_split(team_pools, total=DEBUG_TOTAL_TEAMS, n_test=DEBUG_TEST_TEAMS,
                     source=TEST_SOURCE, seed=SEED):
    rng = random.Random(seed)
    src = list(team_pools[source])
    rng.shuffle(src)
    sample = src[:total]
    return sample[n_test:], sample[:n_test]

if DEBUG_RUN:
    train_teams, test_teams = make_debug_split(TEAM_POOLS)
else:
    train_teams, test_teams = make_split(TEAM_POOLS)
print(f"train teams: {len(train_teams)} | test teams: {len(test_teams)}")

train teams: 40 | test teams: 10


## 6. Model

Per-Pokemon vector =
`[species_emb | species_rdv | ability_emb | item_emb | item_rdv | moveset_emb | moveset_rdv]`.
The masked slot's vector is replaced by a learned mask token. Vectors are projected to `D_MODEL`,
passed through a Transformer encoder (order-free / no positional encoding, since a team is a set),
and the masked slot's output feeds four prediction heads.

In [8]:
class MaskedTeamTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.species_emb = nn.Embedding(N_SPECIES, EMB["species"])
        self.ability_emb = nn.Embedding(N_ABILITY, EMB["ability"])
        self.item_emb    = nn.Embedding(N_ITEM,    EMB["item"])
        self.move_emb    = nn.Embedding(N_MOVE,    EMB["move"])

        self.use_move_attn = USE_MOVE_ATTENTION
        if self.use_move_attn:
            self.move_attn_emb = nn.MultiheadAttention(
                EMB["move"], MOVE_ATTENTION_HEADS, batch_first=True)
            self.move_attn_rdv = nn.MultiheadAttention(
                MOVE_RDV_DIM, MOVE_ATTENTION_HEADS, batch_first=True)

        self.input_dim = (EMB["species"] + SPECIES_RDV_DIM + EMB["ability"]
                          + EMB["item"] + ITEM_RDV_DIM
                          + EMB["move"] + MOVE_RDV_DIM)
        self.mask_token = nn.Parameter(torch.randn(self.input_dim) * 0.02)
        self.input_proj = nn.Linear(self.input_dim, D_MODEL)

        enc = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=N_LAYERS)

        self.head_species = nn.Linear(D_MODEL, N_SPECIES)
        self.head_ability = nn.Linear(D_MODEL, N_ABILITY)
        self.head_item    = nn.Linear(D_MODEL, N_ITEM)
        self.head_moves   = nn.Linear(D_MODEL, N_MOVE)   # multi-label moveset

    def _moveset(self, move_idx, move_rdv):
        # move_idx [B,6,4]  move_rdv [B,6,4,44]
        B = move_idx.size(0)
        me = self.move_emb(move_idx)                       # [B,6,4,move_dim]
        if self.use_move_attn:
            e = me.reshape(B * 6, 4, EMB["move"])
            e, _ = self.move_attn_emb(e, e, e)
            me = e.reshape(B, 6, 4, EMB["move"])
            r = move_rdv.reshape(B * 6, 4, MOVE_RDV_DIM)
            r, _ = self.move_attn_rdv(r, r, r)
            move_rdv = r.reshape(B, 6, 4, MOVE_RDV_DIM)
        return me.mean(dim=2), move_rdv.mean(dim=2)        # [B,6,move_dim],[B,6,44]

    def forward(self, batch):
        sp = self.species_emb(batch["species_idx"])        # [B,6,sp]
        ab = self.ability_emb(batch["ability_idx"])        # [B,6,ab]
        it = self.item_emb(batch["item_idx"])              # [B,6,it]
        ms_emb, ms_rdv = self._moveset(batch["move_idx"], batch["move_rdv"])
        x = torch.cat([sp, batch["species_rdv"], ab,
                       it, batch["item_rdv"], ms_emb, ms_rdv], dim=-1)  # [B,6,input_dim]

        B = x.size(0)
        mpos = batch["mask_pos"]                            # [B]
        x = x.clone()
        x[torch.arange(B), mpos] = self.mask_token

        h = self.encoder(self.input_proj(x))               # [B,6,D_MODEL]
        hm = h[torch.arange(B), mpos]                       # [B,D_MODEL]
        return {
            "species": self.head_species(hm),
            "ability": self.head_ability(hm),
            "item":    self.head_item(hm),
            "moves":   self.head_moves(hm),
        }

## 7. Training & Validation Pipeline

In [9]:
ce  = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

def compute_loss(out, batch):
    return (ce(out["species"], batch["y_species"])
            + ce(out["ability"], batch["y_ability"])
            + ce(out["item"],    batch["y_item"])
            + bce(out["moves"],  batch["y_moves"]))

def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    n = 0
    correct = {"species": 0, "ability": 0, "item": 0}
    move_recall = 0.0
    total_loss = 0.0
    for batch in loader:
        batch = to_device(batch, device)
        out = model(batch)
        total_loss += compute_loss(out, batch).item() * batch["y_species"].size(0)
        for k in correct:
            correct[k] += (out[k].argmax(-1) == batch[f"y_{k}"]).sum().item()
        for i in range(batch["y_moves"].size(0)):
            true_idx = batch["y_moves"][i].nonzero(as_tuple=True)[0]
            k = max(int(len(true_idx)), 1)
            pred_topk = out["moves"][i].topk(k).indices
            hit = len(set(pred_topk.tolist()) & set(true_idx.tolist()))
            move_recall += hit / max(len(true_idx), 1)
        n += batch["y_species"].size(0)
    return {
        "loss": total_loss / max(n, 1),
        "species_acc": correct["species"] / max(n, 1),
        "ability_acc": correct["ability"] / max(n, 1),
        "item_acc":    correct["item"]    / max(n, 1),
        "move_recall": move_recall / max(n, 1),
    }

def train(model, train_ds, test_ds, epochs=EPOCHS, batch_size=BATCH_SIZE,
          lr=LR, device=DEVICE):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    vl = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    for ep in range(1, epochs + 1):
        model.train()
        run = 0.0
        for batch in tl:
            batch = to_device(batch, device)
            opt.zero_grad()
            loss = compute_loss(model(batch), batch)
            loss.backward()
            opt.step()
            run += loss.item() * batch["y_species"].size(0)
        val = evaluate(model, vl, device)
        print(f"epoch {ep:2d} | train_loss {run/len(train_ds):.4f} "
              f"| val_loss {val['loss']:.4f} "
              f"| spc {val['species_acc']:.2f} ab {val['ability_acc']:.2f} "
              f"it {val['item_acc']:.2f} mv {val['move_recall']:.2f}")
    return model

## 8. Debug Run — 50 teams, small config

Smoke test only: 40 train / 10 test, `small` embeddings. Accuracy is not meaningful at this scale —
this just verifies the pipeline runs end to end.

In [10]:
assert EMBED_CONFIG_NAME == "small", "Debug run expects the small config."
set_seed(SEED)

train_ds = MaskedTeamDataset(train_teams, TRAIN_MASK_STRATEGY)
test_ds  = MaskedTeamDataset(test_teams,  EVAL_MASK_STRATEGY)
print(f"datasets -> train {len(train_ds)}, test {len(test_ds)}")

# sanity-check a single sample / forward pass
model = MaskedTeamTransformer()
print("per-Pokemon input dim:", model.input_dim)
_sample = next(iter(DataLoader(train_ds, batch_size=4)))
_out = model(to_device(model.cpu() and _sample, "cpu"))
print("output shapes:", {k: tuple(v.shape) for k, v in _out.items()})

model = train(model, train_ds, test_ds, epochs=EPOCHS)
print("\nFinal:", evaluate(model, DataLoader(test_ds, batch_size=BATCH_SIZE), DEVICE))

datasets -> train 40, test 10
per-Pokemon input dim: 140
output shapes: {'species': (4, 359), 'ability': (4, 269), 'item': (4, 209), 'moves': (4, 682)}


epoch  1 | train_loss 17.8466 | val_loss 15.4575 | spc 0.00 ab 0.20 it 0.00 mv 0.00
epoch  2 | train_loss 16.3575 | val_loss 14.6971 | spc 0.00 ab 0.30 it 0.40 mv 0.03
epoch  3 | train_loss 15.2210 | val_loss 14.1337 | spc 0.00 ab 0.30 it 0.40 mv 0.03
epoch  4 | train_loss 15.1097 | val_loss 13.6297 | spc 0.00 ab 0.20 it 0.40 mv 0.03
epoch  5 | train_loss 14.6143 | val_loss 13.0662 | spc 0.00 ab 0.30 it 0.40 mv 0.07
epoch  6 | train_loss 13.8052 | val_loss 12.4753 | spc 0.00 ab 0.30 it 0.40 mv 0.15
epoch  7 | train_loss 13.3439 | val_loss 11.9925 | spc 0.30 ab 0.30 it 0.40 mv 0.15
epoch  8 | train_loss 12.8957 | val_loss 11.6190 | spc 0.30 ab 0.30 it 0.40 mv 0.15
epoch  9 | train_loss 13.0880 | val_loss 11.4057 | spc 0.10 ab 0.30 it 0.40 mv 0.15
epoch 10 | train_loss 12.7627 | val_loss 11.2181 | spc 0.10 ab 0.20 it 0.40 mv 0.15

Final: {'loss': 11.218063354492188, 'species_acc': 0.1, 'ability_acc': 0.2, 'item_acc': 0.4, 'move_recall': 0.15}


## 9. Saving / Loading the Model

`save_checkpoint` bundles the trained weights together with the architecture config and the
vocabularies, so the model can be rebuilt for inference later without re-deriving anything.
`load_checkpoint` reconstructs it. The debug-trained model is saved below.

In [11]:
import os

def save_checkpoint(model, path="masked_team_transformer.pt"):
    torch.save({
        "state_dict": model.state_dict(),
        "config": {
            "EMB": EMB,
            "SPECIES_RDV_DIM": SPECIES_RDV_DIM,
            "ITEM_RDV_DIM": ITEM_RDV_DIM,
            "MOVE_RDV_DIM": MOVE_RDV_DIM,
            "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS,
            "DIM_FEEDFORWARD": DIM_FEEDFORWARD, "DROPOUT": DROPOUT,
            "USE_MOVE_ATTENTION": USE_MOVE_ATTENTION,
            "MOVE_ATTENTION_HEADS": MOVE_ATTENTION_HEADS,
            "embed_config_name": EMBED_CONFIG_NAME,
        },
        "vocabs": {
            "species": SPECIES_VOCAB, "ability": ABILITY_VOCAB,
            "item": ITEM_VOCAB, "move": MOVE_VOCAB,
        },
    }, path)
    print(f"saved checkpoint -> {path} ({os.path.getsize(path)/1e6:.2f} MB)")

def load_checkpoint(path="masked_team_transformer.pt", device=DEVICE):
    """Rebuilds the model from a checkpoint. Assumes the architecture/vocab
    globals in this notebook match those stored in the checkpoint."""
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = MaskedTeamTransformer()
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    print(f"loaded checkpoint <- {path} "
          f"(trained config: {ckpt['config']['embed_config_name']})")
    return model, ckpt

# Save the debug-trained model, then verify it reloads and matches
save_checkpoint(model, "masked_team_transformer.pt")
_reloaded, _ckpt = load_checkpoint("masked_team_transformer.pt")
_chk = evaluate(_reloaded, DataLoader(test_ds, batch_size=BATCH_SIZE), DEVICE)
print("reloaded model eval:", _chk)

saved checkpoint -> masked_team_transformer.pt (2.05 MB)
loaded checkpoint <- masked_team_transformer.pt (trained config: small)
reloaded model eval: {'loss': 11.218063354492188, 'species_acc': 0.1, 'ability_acc': 0.2, 'item_acc': 0.4, 'move_recall': 0.15}
